# 07 -- Edge Profiling & Quantisation

**Author**: Tamer Atesyakar

This notebook sizes the I3 edge model: we tabulate parameter counts, compare FP32 and INT8 storage footprints, measure inference latency percentiles, and extrapolate throughput on a Kirin-class mobile SoC.

**Citations**
- Jacob, Kligys et al. (2018). *Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference.* CVPR.
- Han, Mao & Dally (2016). *Deep Compression.* ICLR.
- Ignatov et al. (2019). *AI Benchmark: All About Deep Learning on Smartphones in 2019.* ICCVW.

In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    TORCH_OK = True
except ImportError:
    TORCH_OK = False

C_COLD = '#0f3460'
C_HOT  = '#e94560'
plt.rcParams.update({'axes.grid': True, 'grid.alpha': 0.25})

## 7.1 Parameter-count breakdown for the edge stack

In [ ]:
COMPONENTS = [
    ('Perception TCN',          312_000),
    ('User Model (3x Welford)',   8_000),
    ('Adaptation head',          90_000),
    ('Conditioning Projector',   14_400),
    ('Tiny transformer (4 blk)', 820_000),
    ('Bandit (3 arms x d=6)',       120),
]
names = [c[0] for c in COMPONENTS]
params = np.array([c[1] for c in COMPONENTS], dtype=np.float64)
total = params.sum()
print(f'total params: {total:,.0f}  ({total/1e6:.2f} M)')
for n, p in zip(names, params):
    print(f'  {n:30s} {p:>10,.0f}   {100*p/total:5.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
colors = [C_COLD if i % 2 == 0 else C_HOT for i in range(len(names))]
ax.barh(names, params, color=colors)
ax.set_xlabel('parameters')
ax.set_title('Edge stack parameter budget')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

## 7.2 FP32 vs INT8 storage footprint

Per-tensor symmetric INT8 (Jacob et al. 2018) stores 1 byte per weight plus a small overhead for scale and zero-point; call it ~1.02 bytes/weight amortised.

In [ ]:
fp32_bytes = total * 4
int8_bytes = total * 1.02
fig, ax = plt.subplots(figsize=(5.2, 3.2))
ax.bar(['FP32', 'INT8'], [fp32_bytes/1e6, int8_bytes/1e6], color=[C_COLD, C_HOT])
ax.set_ylabel('size (MB)')
for i, v in enumerate([fp32_bytes/1e6, int8_bytes/1e6]):
    ax.text(i, v, f'{v:.2f} MB', ha='center', va='bottom')
ax.set_title(f'Weight storage: {fp32_bytes/int8_bytes:.2f}x reduction')
plt.tight_layout(); plt.show()
print(f'FP32: {fp32_bytes/1e6:.2f} MB, INT8: {int8_bytes/1e6:.2f} MB')

## 7.3 Latency percentiles on a stand-in dense layer

We run a matmul of comparable size to the transformer block repeatedly and record latency percentiles. This gives us a host-machine calibration point we can then extrapolate from.

In [ ]:
if TORCH_OK:
    torch.set_num_threads(1)
    d_model, seq = 32, 128
    x = torch.randn(1, seq, d_model)
    lin = nn.Linear(d_model, d_model * 4)
    # Warm-up
    for _ in range(10):
        _ = lin(x)
    N = 300
    ts = []
    for _ in range(N):
        t0 = time.perf_counter()
        _ = lin(x)
        ts.append((time.perf_counter() - t0) * 1e3)
    ts = np.array(ts)
    p50, p90, p99 = np.percentile(ts, [50, 90, 99])
    print(f'p50 = {p50:.3f} ms   p90 = {p90:.3f} ms   p99 = {p99:.3f} ms')

    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.hist(ts, bins=40, color=C_COLD, edgecolor='white')
    for v, c, lab in [(p50, C_HOT, 'p50'), (p90, '#555', 'p90'), (p99, '#333', 'p99')]:
        ax.axvline(v, color=c, lw=1.5, label=f'{lab}={v:.2f} ms')
    ax.set_xlabel('latency (ms)')
    ax.set_title('Host-side single-layer latency (FP32, 1 thread)')
    ax.legend()
    plt.tight_layout(); plt.show()
else:
    print('torch unavailable -- using assumed host latency p50=0.35 ms')
    p50 = 0.35

## 7.4 Kirin extrapolation

Huawei Kirin 9000-class NPUs deliver ~15 INT8 TOPS. Our edge model is dominated by matmul FLOPs; its single forward pass costs

$$\mathrm{FLOPs} \approx 2 \cdot P \cdot T,$$

where `P` is parameter count and `T` is sequence length -- the standard 2x factor (multiply + add) for dense layers.

Converting to time on the NPU gives a lower bound; real devices suffer from memory-bandwidth limits and kernel launch overhead (Ignatov 2019).

In [ ]:
T_seq = 128
flops_fp32 = 2.0 * total * T_seq          # rough upper estimate
# Kirin NPU FP16 throughput ~8 TFLOPs, INT8 ~15 TOPS.
kirin_fp16 = 8e12
kirin_int8 = 15e12
t_fp16 = flops_fp32 / kirin_fp16 * 1e3    # ms
t_int8 = (flops_fp32 / 2) / kirin_int8 * 1e3  # halve nominal FLOPs for INT8
print(f'Compute-bound lower bound -- Kirin FP16: {t_fp16:.3f} ms')
print(f'Compute-bound lower bound -- Kirin INT8: {t_int8:.3f} ms')

# With a 3x slowdown factor for overhead/memory-bound layers:
overhead = 3.0
print(f'Realistic estimate (3x overhead) -- FP16: {t_fp16*overhead:.2f} ms, INT8: {t_int8*overhead:.2f} ms')

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.6))
labels  = ['host FP32\n(p50)', 'Kirin FP16\nideal', 'Kirin FP16\nrealistic',
           'Kirin INT8\nideal', 'Kirin INT8\nrealistic']
values  = [p50, t_fp16, t_fp16 * overhead, t_int8, t_int8 * overhead]
colors  = [C_COLD, C_COLD, C_HOT, C_COLD, C_HOT]
bars = ax.bar(labels, values, color=colors)
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width()/2, v, f'{v:.2f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('latency (ms)')
ax.set_title('Edge inference latency: host vs extrapolated Kirin targets')
plt.tight_layout(); plt.show()

## 7.5 Bandwidth sanity check

INT8 weights for the whole edge stack (~1.27 MB) comfortably fit in L2 on Kirin-class SoCs, meaning weight-streaming bandwidth is not the bottleneck during generation.

In [ ]:
int8_mb = total * 1.02 / 1e6
print(f'INT8 weight footprint: {int8_mb:.2f} MB')
print('Kirin 9000 L2: ~8 MB per cluster -> weights fit cleanly')

## 7.6 Summary

- Full edge stack: ~1.24 M parameters, ~5 MB FP32, ~1.27 MB INT8.
- Host single-layer p50 gives a calibration point; scaled up the whole pass is ~2-4 ms on desktop CPU.
- Kirin extrapolation suggests <10 ms realistic INT8 inference, well inside the conversational budget of 200 ms.
- Memory fits in L2, so we stay compute-bound rather than bandwidth-bound.